# Creating Species Files for Excited-State Calculations
**by <span style="color:darkgreen">Hannah Kleine</span>, <span style="color:darkgreen">Martí Raya Moreno</span> & <span style="color:darkgreen">Sven Lubeck</span> for [<span style="color:darkgoldenrod">exciting *magnesium*</span>](https://www.exciting-code.org/magnesium)**
<hr style="border:2px solid #DDD"> </hr>

**<span style="color:firebrick">Purpose</span>**: In this tutorial, we explain how to generate a species file containing a basis appropriate for excited-state calculations.

<hr style="border:2px solid #DDD"> </hr>

<div class="alert alert-block alert-warning">

**Table of Contents**
    
[0. Before starting](#0)

[1. Importance of specialised excited-state basis sets](#1)

[2. Preparing the Ground-State Species File](#2)

[3. How to add high-energy basis functions](#3)

  - [i) Activate the LO‑recommendation tool](#4)
  - [ii) Selecting only custom APWs and local orbitals not yet included in the basis](#5)

[4. Summary & Next Steps](#6)

</div>

<a id='0'></a>
<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">0. Before Starting</span>

**<span style="color:firebrick">Read the following paragraphs before starting with the rest of this tutorial!</span>**

Before running any Jupyter tutorials, please refer to the **`00_before_starting.md`** document on how to correctly set up the environment. This only needs to be done once. After which, the **venv** can be (re)activated from **`exciting`**'s root directory:

<div style="background-color: rgb(224, 224, 224);">

```bash
source $EXCITINGROOT/tools/excitingjupyter/venv/excitingvenv/bin/activate
```

</div>

Here is a list of the scripts found within the **`excitingscripts`** module which are relevant for this tutorial with a short description.

* **`excitingscripts.setup.excited_state_species_files`**: Python module for generating species files for excited-state calculations.


As a first step, create a run directory for the notebook.

In [ ]:
%%bash
mkdir -p run_excited_state_species_file_generation

<a id='1'></a>
<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">1. Importance of specialised excited-state basis sets</span>
In all-electron methods such as **(L)APW+lo**, the **quality and completeness of the basis set** are absolutely critical—especially when considering that **excited-state properties depend more strongly on the basis than ground-state ones**.

Ground-state (GS) calculations mainly require an accurate description of *occupied* and low-lying *unoccupied* states.
However, excited-state calculations (GW, BSE, RPA, etc.) explicitly involve **empty bands**, often extending **tens of eV above the Fermi level**. These states are poorly represented even by a basis that is over-converged for the GS, to the point that the results become unreliable or even qualitatively wrong.

➡️ **Solution:** add **extra Local Orbitals (LOs)** at **high energies** to systematically improve basis flexibility and describe states over the relevant energy range.

<a id='2'></a>
<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">2. Preparing the Ground-State Species File</span>

Before generating a species file for excited-state calculations, an appropriate ground-state species file must first be prepared.
It is important that this file has already been fully optimized for a ground-state calculation. 

⚠️ <span style="color:firebrick"> **Note:**</span> 
The script used later in this tutorial to add high-energy basis functions cannot handle species files containing explicit trial energies
for custom APW basis functions or LOs. If you have such a ground-state species file, convert it to use principal quantum numbers instead of trial energies (see [Understanding the exciting Species Files](../../01_getting_started/understanding_the_exciting_species_files.ipynb)), or add the high-energy basis functions by hand using the LO recommendation described in the next section.

Here you can see an example ground-state species file for Silicon (Si.xml):

<span class="ground_state_species_file"></span>
```xml
<?xml version="1.0" ?>
<spdb>
	<sp chemicalSymbol="Si" name="silicon" z="-14.0" mass="51196.73454">
		<muffinTin rmin="1e-06" radius="2.0" rinf="24.976" radialmeshPoints="600"/>
		<atomicState n="1" l="0" kappa="1" occ="2.0" core="true"/>
		<atomicState n="2" l="0" kappa="1" occ="2.0" core="false"/>
		<atomicState n="2" l="1" kappa="1" occ="2.0" core="false"/>
		<atomicState n="2" l="1" kappa="2" occ="4.0" core="false"/>
		<atomicState n="3" l="0" kappa="1" occ="2.0" core="false"/>
		<atomicState n="3" l="1" kappa="1" occ="1.0" core="false"/>
		<atomicState n="3" l="1" kappa="2" occ="1.0" core="false"/>
		<basis>
			<default type="lapw" trialEnergy="0.15" searchE="false"/>
			<custom l="0" type="lapw" n="3" searchE="false"/>
			<custom l="1" type="lapw" n="3" searchE="false"/>
			<lo l="0">
				<wf matchingOrder="0" searchE="false" n="3"/>
				<wf matchingOrder="0" searchE="false" n="2"/>
			</lo>
			<lo l="1">
				<wf matchingOrder="0" searchE="false" n="3"/>
				<wf matchingOrder="0" searchE="false" n="2"/>
			</lo>
			<lo l="0">
				<wf matchingOrder="0" searchE="false" n="2"/>
				<wf matchingOrder="1" searchE="false" n="2"/>
			</lo>
			<lo l="0">
				<wf matchingOrder="0" searchE="false" n="3"/>
				<wf matchingOrder="1" searchE="false" n="3"/>
			</lo>
			<lo l="1">
				<wf matchingOrder="0" searchE="false" n="2"/>
				<wf matchingOrder="1" searchE="false" n="2"/>
			</lo>
			<lo l="1">
				<wf matchingOrder="0" searchE="false" n="3"/>
				<wf matchingOrder="1" searchE="false" n="3"/>
			</lo>
		</basis>
	</sp>
</spdb>
```


It contains a basic APW basis that describes all valence states and, in addition, low-energy LOs that describe semicore states and further improve the description of valence states.

⚠️ <span style="color:firebrick"> **Note:**</span>  If you need help generating such a ground-state species file, have a look at the related [tutorial on ground-state basis set generation](../../05_tools_and_packages/tutorial_customizable_species_files.ipynb).

<a id='3'></a>
<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">3. How to add high-energy basis functions</span>

Starting from your ground-state species file, we will now guide you through the process of systematically adding high-energy basis functions.

<a id='4'></a>
#### <span style="color:#15317E">i) Activate the LO‑recommendation tool</span>

To determine up to which principal quantum number *n* local orbitals need to be added for each *l*-channel, we can use the LO-recommendation tool included in **exciting**. When activated, this tool uses the **Wigner-Seitz rules** to compute appropriate **trial energies** for user-defined combinations of *l*-channels and node numbers, and writes the **trial energy** for each (*l, n*) combination to `LO_RECOMMENDATION.OUT`. This is done for every species included in the ground-state calculation.

Starting from an input file for a standard ground-state calculation, add the <code><span style="color:green">lorecommendation</span></code> element to the <code><span style="color:green">groundstate</span></code> element.


<span class="LO_recommendation_input"></span>
```xml
<input>
 
   <title>Silicon</title>
 
   <structure speciespath=".">
      <crystal>
         <basevect>5.13 5.13 0.00</basevect>
         <basevect>5.13 0.00 5.13</basevect>
         <basevect>0.00 5.13 5.13</basevect>
      </crystal>
      <species speciesfile="Si.xml" rmt="2.1">
         <atom coord="0.00 0.00 0.00"></atom>
         <atom coord="0.25 0.25 0.25"></atom>
      </species>
   </structure>
 
   <groundstate
      do="fromscratch"
      rgkmax="7.0"
      ngridk="3 3 3"
      xctype="LDA_PW"
      >
     <lorecommendation
	 lmaxlo="4"
	 nodesmaxlo="10"/>
   </groundstate>
 
</input>
```

**Meaning of the parameters:**

- <code><span style="color:mediumblue">lmaxlo</span>=<span style="color:firebrick">"4"</span></code> — scan angular momenta \(l = 0,1,2,3,4\)
- <code><span style="color:mediumblue">nodesmaxlo</span>=<span style="color:firebrick">"10"</span></code> — include radial solutions with up to 10 nodes

Now we can write the input and species file to our run directory:

In [ ]:
import os
from excitingjupyter.utilities import get_input_xml_from_notebook   

# Extract input file content from this notebook:
input_str = get_input_xml_from_notebook("species_files_for_excited_states", "LO_recommendation_input")

# Write out the input as an XML file:
with open('./run_excited_state_species_file_generation/input.xml', "w") as fid:
    fid.write(input_str)
    
# Extract species file content from this notebook:
species_str = get_input_xml_from_notebook("species_files_for_excited_states", "ground_state_species_file")

# Write out the species as an XML file:
with open('./run_excited_state_species_file_generation/Si.xml', "w") as fid:
    fid.write(species_str)

In [ ]:
%%bash
cd run_excited_state_species_file_generation
time python3 -m excitingscripts.execute.single -f input.xml
cd ..

Once the ground-state run finishes, we can have a look at the `LO_RECOMMENDATION.OUT` file which contains a recommended trial energy for each scanned (*l, n*) combination. 

In [ ]:
%%bash
cd run_excited_state_species_file_generation
cat LO_RECOMMENDATION.OUT
cd ..

In the next step, we use these values to decide which basis functions to add.

<a id='5'></a>
#### <span style="color:#15317E">ii) Selecting only custom APWs and local orbitals not yet included in the basis</span>

You should **not** duplicate basis functions that are already present in your ground-state species file, since duplicates can cause linear dependencies. To help avoid this, use the excited-state species-file generation script from the `excitingscripts` package.

Rather than inserting basis functions manually, the script:
1. reads an existing ground-state species file,
2. parses `LO_RECOMMENDATION.OUT`,
3. determines all recommended principal quantum numbers whose recommended energies lie below a user-defined energy threshold,
4. inserts all missing basis functions, while avoiding duplicates and basis functions that are very likely to cause linear dependencies,
5. writes the resulting excited-state species file.

**Added basis functions**

The script adds the following basis functions for all states up to the given energy threshold:

**for *l*-channel not yet part of the ground-state basis**

- one custom LAPW at the lowest possible principal quantum number (`n = l + 1`)
- one local orbital with matching orders `(0, 0)` bridging each pair of adjacent required `n` states (skipped for the very first, lowest state, since there is no lower state to bridge from)
- local orbitals with matching orders `(0, 1)`, `(1, 2)`, `(2, 3)` up to a given maximum matching order for every required (l, n) combination

**for *l*-channel already part of the ground-state basis**

- the same `(0, 0)` bridging LOs
- higher-matching-order LOs as above, continuing on from the highest `n` already present in the file

**Command-Line Usage**

To use the script, run:

In [ ]:
%%bash
cd run_excited_state_species_file_generation
python3 -m excitingscripts.setup.excited_state_species_files Si -e 80 --path-xml . --path-lo . --output .
cd ..

**Arguments:**

- <code><span style="color:mediumblue">-e</span> <span style="color:green"> 80 </span></code> — Add local orbitals up to a **trial energy** of 80 Ha
- <code><span style="color:mediumblue">--path-xml .</span></code> — use the ground-state species file from `.`
- <code><span style="color:mediumblue">--path-lo .</span></code> — use the `LO_RECOMMENDATION.OUT` file from `.`
- <code><span style="color:mediumblue">--output .</span></code> — write the excited-state species file to `.`


The script reports each added basis function and writes `Si_excited.xml`. For this example, a successful result contains new custom LAPWs for \(l=2,3,4\), no duplicate local orbitals, and high-energy local orbitals through \(n=9,9,10,10,11\) for \(l=0,1,2,3,4\), respectively. The generated species file can then be inspected directly.

In [ ]:
%%bash
cd run_excited_state_species_file_generation
cat Si_excited.xml
cd ..


You can also specify the following optional arguments:
- <code><span style="color:mediumblue">--max-matching-order</span></code> or <code><span style="color:mediumblue">-mo</span></code> — highest matching order used for additional local orbitals. Accepted values are `0`, `1`, `2`, and `3`; the default is `1`.
- <code><span style="color:mediumblue">--search-e</span></code> — If set, newly added basis functions have <code><span style="color:mediumblue">searchE=</span><span style="color:firebrick">"true"</span></code> instead of <code><span style="color:firebrick">"false"</span></code> 
- <code><span style="color:mediumblue">--keep-lin-dep</span></code> — If set, disables automatic skipping of linearly dependent high-energy local orbitals

<div class="alert alert-block alert-danger">

⚠️ **General remarks**

There is no universal default value for the trial energy threshold, since it is highly material-dependent. Both this threshold and the maximum `l`-channel considered (`lmaxlo`) are convergence parameters that should be tested for your specific system. Also keep in mind that the convergence of the number of unoccupied states (`nempty`) can depend on the local-orbital set used.
    
Local orbitals can only be added for the trial-energy range represented in `LO_RECOMMENDATION.OUT`. If <code><span style="color:mediumblue">nodesmaxlo</span></code> is too small to reach your trial-energy threshold, increase it and rerun the LO recommendation.

</div>

<a id='6'></a>
<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">4. Summary & Next Steps</span>

In this tutorial we covered how to extend a ground-state species file with high-energy local orbitals suitable for excited-state calculations:

1. Start from a fully optimized **ground-state species file**.
2. Run a ground-state calculation with an `<lorecommendation>` block to generate `LO_RECOMMENDATION.OUT`.
3. Use the excited-state species-file generation script to add missing custom APWs and local orbitals up to a chosen energy threshold. The script avoids duplicates and skips basis functions that are likely to cause linear dependencies.
4. Converge your results with respect to the energy threshold, `lmaxlo`, and `nempty`.

With your new excited-state species file in hand, you are ready to proceed with excited-state calculations such as GW or BSE. 